# Grid search - BETO head

In [ ]:

!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.beto import build_beto
from betohumor.hyperparam_search import run_search
from betohumor.utils import save_metrics
from betohumor.plots import plot_training_curves, plot_grid_search_comparison
from itertools import product

## 1. Datos y configuraciones

In [3]:
data_config  = DataConfig(data_path = "../data/raw/haha_2019_train.csv")
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

Train: 19197 | Val: 2397 | Test: 2400


## 2. Definición de la grilla


In [7]:
LR_VALUES = [1e-4, 5e-4, 1e-3, 5e-3]
WEIGHT_DECAYS = [0.0, 0.01, 0.05, 0.1]

search_grid = [
    {
        'learning_rate': lr,
        'weight_decay': wd
    }
    for  lr,wd in product(LR_VALUES, WEIGHT_DECAYS)
]



search_grid

[{'learning_rate': 0.0001, 'weight_decay': 0.0},
 {'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'learning_rate': 0.0001, 'weight_decay': 0.1},
 {'learning_rate': 0.0005, 'weight_decay': 0.0},
 {'learning_rate': 0.0005, 'weight_decay': 0.01},
 {'learning_rate': 0.0005, 'weight_decay': 0.05},
 {'learning_rate': 0.0005, 'weight_decay': 0.1},
 {'learning_rate': 0.001, 'weight_decay': 0.0},
 {'learning_rate': 0.001, 'weight_decay': 0.01},
 {'learning_rate': 0.001, 'weight_decay': 0.05},
 {'learning_rate': 0.001, 'weight_decay': 0.1},
 {'learning_rate': 0.005, 'weight_decay': 0.0},
 {'learning_rate': 0.005, 'weight_decay': 0.01},
 {'learning_rate': 0.005, 'weight_decay': 0.05},
 {'learning_rate': 0.005, 'weight_decay': 0.1}]

## 3. Correr el search

In [11]:
results = run_search( lambda _: build_beto(beto_config),search_grid, beto_config, data_config,
    df_train, df_val, tokenizer, seed=data_config.seed,
    output_dir_prefix='results/checkpoints/search_baseline',
)

print('\n=== RESULTADOS (ordenados por Macro F1) ===')
for r in results:
    print(f"  {r['params']} -> Macro F1: {r['macro_f1']:.4f}")


=== learning_rate0.0001_weight_decay0.0 ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 63731.42it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

Baseline — entrenables: 592,130 / 109,852,418


/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## 4. Curvas de entrenamiento 

In [ ]:
for r in results:
    print(f"\n {r['run_name']} (Macro F1: {r['macro_f1']:.4f})")
    plot_training_curves(r['history'])

## 5. Comparación entre resultados


In [ ]:
labels = [r['run_name'] for r in results]
scores = [r['macro_f1'] for r in results]
plot_grid_search_comparison(labels, scores, title='Grid search - Macro F1 en validación')

## 6. Guardar métricas


In [ ]:
all_metrics = {
    r['run_name']: {'macro_f1': r['macro_f1'], 'params': r['params']}
    for r in results
}

save_metrics(all_metrics, folder_name='hyperparams_search_beto_head')

## 6. Análisis
TODO!!